# Coadd Flux Check — sources @ 90/150 GHz

目的：在 big coadd map 中，用 CSV 里给的 (xcentroid, ycentroid) pixel 位置直接读取每个源的 flux level，画 cutout grid + 输出 summary。

- Map: `data/coadd/galaxy_3yr_pc_gc_v2_{90,150}.fits`，5280×2880，ZEA 投影
- 单位：raw data 是 mK_CMB（FITS header `UNITS=Tcmb`），除以 `G3Units.uK = 1e-3` → μK
- Cutout: 50×50 pixel（≈ 12.5′ × 12.5′，pix scale 15″）
- 显示范围：vmin/vmax = ±1000 μK

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', '..', 'src'))   # notebooks/diagnostics/ -> repo/src

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from paths import DATA, OUT
from units import spt_to_uk, cbar_label
from cutouts import open_coadd, coadd_cutout
from plotting import coadd_grid

CSV_PATH = os.path.join(DATA, '2024_2023_yearly_full.csv')

HALF = 25            # cutout half-size (pixels)
VLIM = 1000          # color scale ±μK

In [ ]:
# Load CSV — strip trailing spaces from column names, drop blank rows
raw = pd.read_csv(CSV_PATH)
raw.columns = [c.strip() for c in raw.columns]
df = raw[['id', 'band', 'xcentroid', 'ycentroid', 'max_value', 'snr_max']].copy()
df['id'] = df['id'].astype(str).str.strip()
df['band'] = df['band'].astype(str).str.strip()
df = df.dropna(subset=['xcentroid', 'ycentroid']).reset_index(drop=True)
df['xcentroid'] = df['xcentroid'].astype(float)
df['ycentroid'] = df['ycentroid'].astype(float)
print(f'Loaded {len(df)} sources')
df.head()

In [ ]:
# Open both coadd maps once (raw values are mK_CMB)
maps = {'90': open_coadd('90'), '150': open_coadd('150')}
print({b: m.shape for b, m in maps.items()})

## 1. 单源 sanity check（你给的例子：x=4262, y=499）

In [ ]:
x, y = 4262, 499
fig, ax = plt.subplots(1, 2, figsize=(8, 4))
for a, b in zip(ax, ['90', '150']):
    cut = spt_to_uk(maps[b][y-HALF:y+HALF, x-HALF:x+HALF])
    im = a.imshow(cut, vmin=-VLIM, vmax=VLIM, origin='lower', cmap='RdBu_r')
    a.set_title(f'{b} GHz   peak={np.nanmax(np.abs(cut)):.0f} μK')
    a.axhline(HALF, color='k', lw=0.4); a.axvline(HALF, color='k', lw=0.4)
    plt.colorbar(im, ax=a, fraction=0.046, label=cbar_label(f'SPT-{b}', 'μK'))
fig.suptitle(f'Sanity check: pixel ({x}, {y}) → SPT3G_J170334.1-355347.0')
fig.tight_layout()

## 2. 全部源 cutout grid（4×8 layout）

In [ ]:
coadd_grid({'90 GHz': maps['90']}, df, half=HALF, vlim=VLIM);

In [ ]:
coadd_grid({'150 GHz': maps['150']}, df, half=HALF, vlim=VLIM);

## 3. Summary table — 每个源在 coadd 里的 flux level

在 cutout 中心 ±2 pix 小窗口（5×5 ≈ 75″）取 peak（绝对值最大、保留正负号）和 mean，作为 "flux level" 的代表。

In [ ]:
INNER = 2  # ±2 pix → 5x5 window for central stats

rows = []
for _, r in df.iterrows():
    entry = {'id': r['id'], 'snr_max_csv': r['snr_max'],
             'x': int(round(r['xcentroid'])), 'y': int(round(r['ycentroid']))}
    for b in ['90', '150']:
        cut = spt_to_uk(coadd_cutout(maps[b], r['xcentroid'], r['ycentroid'], half=HALF))
        center = cut[HALF-INNER:HALF+INNER+1, HALF-INNER:HALF+INNER+1]
        flat = center[np.isfinite(center)]
        if flat.size == 0:
            entry[f'peak_{b}'] = np.nan; entry[f'mean_{b}'] = np.nan
        else:
            i_pk = np.argmax(np.abs(flat))
            entry[f'peak_{b}'] = round(float(flat[i_pk]), 1)
            entry[f'mean_{b}'] = round(float(np.nanmean(center)), 1)
    rows.append(entry)

summary = pd.DataFrame(rows)
summary['ratio_150_90'] = (summary['peak_150'] / summary['peak_90']).round(2)
summary

In [ ]:
# Average flux level across all sources (signed)
print(f'Mean over {len(summary)} sources (signed peak in central 5x5, μK):')
print(f"  90  GHz: mean={summary['peak_90'].mean():.1f}  median={summary['peak_90'].median():.1f}  "
      f"|mean|={summary['peak_90'].abs().mean():.1f}")
print(f"  150 GHz: mean={summary['peak_150'].mean():.1f}  median={summary['peak_150'].median():.1f}  "
      f"|mean|={summary['peak_150'].abs().mean():.1f}")

out_path = os.path.join(OUT, 'coadd_flux_check.csv')
summary.to_csv(out_path, index=False)
print(f'\nSaved -> {out_path}')